# Phase 4 — Single Tree vs Ensemble: Decision Tree vs Random Forest
## Dataset: GSE6575 — Gene Expression in Blood of Children with ASD
**Dr. Nourhène Ben Rabah — Introduction to Machine Learning**

This phase trains a **Random Forest** ensemble and compares it against a single
**Decision Tree** — the *base learner* that a Random Forest is built from.
Putting the two side by side isolates exactly what the ensemble adds.

- **Part 1** — Principle of the Decision Tree and of Random Forest
- **Part 2** — Training and evaluation of Random Forest on the test data
- **Part 3** — Comparison with a single Decision Tree
- **Part 4** — Advantages and limitations: single tree vs ensemble

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    StratifiedKFold, cross_val_score, train_test_split
)
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("All libraries loaded successfully.")

---
## Load Data and Reproduce Phase 3 Setup

We reload the same prepared data and reproduce the **exact same** train/test split
and PCA used in Phase 3, so that the comparison is fair.


In [ ]:
# ── Load prepared data (same files as Phase 3) ───────────────────────────────
X = pd.read_csv('X_prepared.csv', index_col=0)
y = pd.read_csv('y_labels.csv', index_col=0).squeeze()

print(f"Feature matrix X : {X.shape}  (samples x probes)")
print(f"Label vector y   : {y.shape}")
print()
print("Class distribution:")
for cls, n in y.value_counts().items():
    print(f"  {cls:<35} {n} samples")

In [ ]:
# ── Reproduce Phase 3 train/test split and PCA exactly ───────────────────────
# Same random_state=42, test_size=0.20, stratify=y as Phase 3.
# This ensures we compare both models on the IDENTICAL test samples.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

pca = PCA(n_components=0.95, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train)
X_test_pca  = pca.transform(X_test)

print(f"Train / Test split  : {X_train.shape[0]} / {X_test.shape[0]} samples")
print(f"PCA components kept : {pca.n_components_}  (explaining 95% of variance)")
print(f"X_train after PCA   : {X_train_pca.shape}")
print(f"X_test  after PCA   : {X_test_pca.shape}")

---
## Part 1 — Principle: Decision Tree and Random Forest

### 1.1 How a single Decision Tree works

A **Decision Tree** partitions the feature space with a sequence of yes/no
questions. At each node it chooses the `(feature, threshold)` pair that best
separates the classes (highest Gini gain) and keeps splitting until the leaves
are pure or a stopping rule is reached.

A single, unpruned tree has **low bias** but **high variance**: it can fit the
training data almost perfectly (often 100% training accuracy), yet a small change
in the data can produce a very different tree, so it generalises poorly. Its one
big strength is **interpretability** — the whole decision path is human-readable.

### 1.2 How Random Forest works

Random Forest is an **ensemble** that builds many decision trees and combines
them by **majority vote**. Two sources of randomness decorrelate the trees:

```
For each tree t = 1 ... n_estimators:
    1. Draw a bootstrap sample (n samples with replacement) from training data
    2. At each node:
       a. Draw max_features random features
       b. Find the BEST split threshold among those candidates (Gini gain)
    3. Grow the tree until min_samples_leaf / max_depth is reached
Final prediction = majority vote across all trees
```

Averaging over many diverse trees **reduces variance** without raising bias much
— which is exactly the weakness of the single tree that it fixes.

### 1.3 Decision Tree vs Random Forest — key differences

| Property | Decision Tree | Random Forest |
|---|---|---|
| **Number of trees** | 1 | 100 (`n_estimators`) |
| **Training data per tree** | Full training set | Bootstrap sample (with replacement) |
| **Features at each split** | All features | Random subset (`max_features`) |
| **Bias** | Low | Low |
| **Variance** | **High** | **Reduced** (averaging) |
| **Overfitting risk** | High | Lower |
| **Interpretability** | **High** (one readable tree) | Low (black box of 100 trees) |
| **Training speed** | Fast | Moderate |
| **Best for** | Quick, explainable baseline | Stable, accurate predictions |

### 1.4 Why compare against a single Decision Tree?

The Decision Tree is the **base learner** of the Random Forest — a forest is
literally a collection of trees like it. Comparing the two therefore isolates the
single thing the ensemble adds: **variance reduction through bootstrap aggregation
(bagging) and feature randomness**. If the forest clearly beats one tree, we have
direct evidence that ensembling helps on this small, high-dimensional dataset, and
the gap between the tree's training and test accuracy makes its overfitting
visible.

---
## Part 2 — Training and Evaluation of Random Forest

### 2.1 Hyperparameters


In [ ]:
# ── Random Forest model ──────────────────────────────────────────────────────
# We use the same hyperparameters as Phase 3's Extra Trees for a fair comparison.
# The only difference is the algorithm itself.

rf_model = RandomForestClassifier(
    n_estimators=100,       # same as Phase 3
    max_features='sqrt',    # same as Phase 3
    class_weight='balanced',# same as Phase 3 — handles MR/DD imbalance
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_model.fit(X_train_pca, y_train)
print("Random Forest model trained successfully.")
print(f"  n_estimators : {rf_model.n_estimators}")
print(f"  max_features : {rf_model.max_features}")
print(f"  n_features   : {X_train_pca.shape[1]} PCA components")

### 2.2 Training Accuracy

In [ ]:
y_train_pred_rf = rf_model.predict(X_train_pca)
train_acc_rf = accuracy_score(y_train, y_train_pred_rf)
train_f1_rf  = f1_score(y_train, y_train_pred_rf, average='weighted')

print("=== Random Forest — Training Results ===")
print(f"  Training Accuracy        : {train_acc_rf:.4f}")
print(f"  Training F1 (weighted)   : {train_f1_rf:.4f}")
print()
print("Note: a single unpruned Decision Tree typically reaches 1.0 training")
print("accuracy (it memorises the data). Random Forest often does not, because")
print("bootstrap sampling means each tree sees only part of the data — an early")
print("sign of the forest's lower variance.")

### 2.3 Cross-Validation (5-Fold Stratified)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Pipeline: PCA inside CV to avoid data leakage (same as Phase 3)
rf_pipeline = Pipeline([
    ('pca', PCA(n_components=0.95, random_state=RANDOM_STATE)),
    ('clf', RandomForestClassifier(
        n_estimators=100,
        max_features='sqrt',
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_cv_acc = cross_val_score(rf_pipeline, X, y, cv=skf,
                             scoring='accuracy', n_jobs=-1)
rf_cv_f1  = cross_val_score(rf_pipeline, X, y, cv=skf,
                             scoring='f1_weighted', n_jobs=-1)

print("=== Random Forest — 5-Fold Cross-Validation ===")
print(f"  Accuracy per fold  : {[f'{s:.3f}' for s in rf_cv_acc]}")
print(f"  Mean Accuracy      : {rf_cv_acc.mean():.4f} +/- {rf_cv_acc.std():.4f}")
print()
print(f"  F1 per fold        : {[f'{s:.3f}' for s in rf_cv_f1]}")
print(f"  Mean F1 (weighted) : {rf_cv_f1.mean():.4f} +/- {rf_cv_f1.std():.4f}")

### 2.4 Test Set Evaluation

In [ ]:
y_pred_rf = rf_model.predict(X_test_pca)
test_acc_rf = accuracy_score(y_test, y_pred_rf)
test_f1_rf  = f1_score(y_test, y_pred_rf, average='weighted')

print("=== Random Forest — Test Set Results ===")
print(f"  Test Accuracy      : {test_acc_rf:.4f}")
print(f"  Test F1 (weighted) : {test_f1_rf:.4f}")
print(f"  Test samples       : {X_test.shape[0]}")

### 2.5 Classification Report

In [ ]:
print("=== Classification Report — Random Forest (Test Set) ===")
print(classification_report(y_test, y_pred_rf, zero_division=0))

### 2.6 Confusion Matrix

In [ ]:
labels = sorted(y.unique())
cm_rf = confusion_matrix(y_test, y_pred_rf, labels=labels)

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=labels).plot(
    ax=ax, cmap='Blues', colorbar=True
)
ax.set_title('Confusion Matrix — Random Forest (Test Set)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

### 2.7 Feature Importance — Top PCA Components

In [ ]:
importances_rf = rf_model.feature_importances_
top_n  = min(20, len(importances_rf))
top_idx = np.argsort(importances_rf)[::-1][:top_n]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(top_n), importances_rf[top_idx],
       color='mediumseagreen', edgecolor='white')
ax.set_xticks(range(top_n))
ax.set_xticklabels([f'PC{i+1}' for i in top_idx], rotation=45, ha='right')
ax.set_xlabel('PCA Component')
ax.set_ylabel('Feature Importance (Gini)')
ax.set_title('Top 20 Most Important PCA Components — Random Forest')
plt.tight_layout()
plt.show()

---
## Part 3 — Comparison: Random Forest vs a Single Decision Tree

We now train a single **Decision Tree** on the **identical** setup (same split,
same PCA, same class weighting) so that every number is directly comparable and
the only difference is *single tree vs ensemble*.

In [ ]:
# ── Train a single Decision Tree (the base learner) ──────────────────────────
# Same class_weight and random_state as the Random Forest, so the ONLY
# difference is: one fully-grown tree vs an ensemble of 100 trees.
dt_model = DecisionTreeClassifier(
    class_weight='balanced',   # same imbalance handling as RF
    random_state=RANDOM_STATE
)
dt_model.fit(X_train_pca, y_train)

# Training accuracy (exposes overfitting)
y_train_pred_dt = dt_model.predict(X_train_pca)
train_acc_dt = accuracy_score(y_train, y_train_pred_dt)

# Test performance
y_pred_dt   = dt_model.predict(X_test_pca)
test_acc_dt = accuracy_score(y_test, y_pred_dt)
test_f1_dt  = f1_score(y_test, y_pred_dt, average='weighted')

# Cross-validation with the same leakage-safe pipeline (PCA inside CV)
dt_pipeline = Pipeline([
    ('pca', PCA(n_components=0.95, random_state=RANDOM_STATE)),
    ('clf', DecisionTreeClassifier(
        class_weight='balanced', random_state=RANDOM_STATE
    ))
])
dt_cv_acc = cross_val_score(dt_pipeline, X, y, cv=skf,
                            scoring='accuracy', n_jobs=-1)
dt_cv_f1  = cross_val_score(dt_pipeline, X, y, cv=skf,
                            scoring='f1_weighted', n_jobs=-1)

print("Single Decision Tree trained successfully.")
print(f"  Tree depth        : {dt_model.get_depth()}")
print(f"  Number of leaves  : {dt_model.get_n_leaves()}")
print(f"  Training accuracy : {train_acc_dt:.4f}  (RF train acc: {train_acc_rf:.4f})")
print(f"  Test accuracy     : {test_acc_dt:.4f}")

### 3.1 Visualising the single Decision Tree

The biggest practical advantage of one tree over a forest is that you can
actually **read it**. Below are the top levels of the trained tree (display
truncated to depth 3 for legibility — the full tree is deeper, which is why it
overfits). A Random Forest contains 100 such trees and cannot be inspected this
way.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(
    dt_model,
    max_depth=3,                       # truncate the DISPLAY only (model is deeper)
    feature_names=[f'PC{i+1}' for i in range(X_train_pca.shape[1])],
    class_names=[str(c) for c in dt_model.classes_],
    filled=True, rounded=True, fontsize=8, impurity=False, ax=ax
)
ax.set_title('Single Decision Tree — top 3 levels (display-truncated)')
plt.tight_layout()
plt.show()

### 3.2 Metrics Comparison Table

In [ ]:
comparison = pd.DataFrame({
    'Metric': [
        'Training Accuracy',
        'CV Accuracy (mean)',
        'CV Accuracy (std)',
        'CV F1 weighted (mean)',
        'CV F1 weighted (std)',
        'Test Accuracy',
        'Test F1 weighted',
    ],
    'Decision Tree': [
        f'{train_acc_dt:.4f}',
        f'{dt_cv_acc.mean():.4f}',
        f'{dt_cv_acc.std():.4f}',
        f'{dt_cv_f1.mean():.4f}',
        f'{dt_cv_f1.std():.4f}',
        f'{test_acc_dt:.4f}',
        f'{test_f1_dt:.4f}',
    ],
    'Random Forest': [
        f'{train_acc_rf:.4f}',
        f'{rf_cv_acc.mean():.4f}',
        f'{rf_cv_acc.std():.4f}',
        f'{rf_cv_f1.mean():.4f}',
        f'{rf_cv_f1.std():.4f}',
        f'{test_acc_rf:.4f}',
        f'{test_f1_rf:.4f}',
    ]
})

print("=== Performance Comparison: Decision Tree vs Random Forest ===")
print(comparison.to_string(index=False))
print()
print("Training-accuracy gap (overfitting indicator):")
print(f"  Decision Tree : {train_acc_dt:.4f} train  ->  {test_acc_dt:.4f} test"
      f"   (drop {train_acc_dt - test_acc_dt:+.4f})")
print(f"  Random Forest : {train_acc_rf:.4f} train  ->  {test_acc_rf:.4f} test"
      f"   (drop {train_acc_rf - test_acc_rf:+.4f})")

### 3.3 Visual Comparison — CV Accuracy per Fold

In [ ]:
folds = [f'Fold {i+1}' for i in range(5)]
x = np.arange(len(folds))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CV accuracy per fold
axes[0].bar(x - width/2, dt_cv_acc, width, label='Decision Tree',
            color='indianred', edgecolor='white')
axes[0].bar(x + width/2, rf_cv_acc, width, label='Random Forest',
            color='mediumseagreen', edgecolor='white')
axes[0].axhline(dt_cv_acc.mean(), color='indianred',
                linestyle='--', alpha=0.7, label=f'DT mean={dt_cv_acc.mean():.3f}')
axes[0].axhline(rf_cv_acc.mean(), color='mediumseagreen',
                linestyle='--', alpha=0.7, label=f'RF mean={rf_cv_acc.mean():.3f}')
axes[0].set_xticks(x)
axes[0].set_xticklabels(folds)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel('Accuracy')
axes[0].set_title('CV Accuracy per Fold')
axes[0].legend(fontsize=8)

# Summary bar chart
metrics   = ['CV Accuracy', 'CV F1', 'Test Accuracy', 'Test F1']
dt_values = [dt_cv_acc.mean(), dt_cv_f1.mean(), test_acc_dt, test_f1_dt]
rf_values = [rf_cv_acc.mean(), rf_cv_f1.mean(), test_acc_rf, test_f1_rf]
x2 = np.arange(len(metrics))

axes[1].bar(x2 - width/2, dt_values, width, label='Decision Tree',
            color='indianred', edgecolor='white')
axes[1].bar(x2 + width/2, rf_values, width, label='Random Forest',
            color='mediumseagreen', edgecolor='white')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(metrics, rotation=15, ha='right')
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel('Score')
axes[1].set_title('Overall Performance Comparison')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

### 3.4 Side-by-Side Confusion Matrices

In [ ]:
cm_dt = confusion_matrix(y_test, y_pred_dt, labels=labels)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ConfusionMatrixDisplay(confusion_matrix=cm_dt, display_labels=labels).plot(
    ax=axes[0], cmap='Reds', colorbar=False)
axes[0].set_title('Single Decision Tree')
plt.setp(axes[0].get_xticklabels(), rotation=30, ha='right')

ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=labels).plot(
    ax=axes[1], cmap='Greens', colorbar=False)
axes[1].set_title('Random Forest')
plt.setp(axes[1].get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.show()

### 3.5 Feature Importance Comparison

In [ ]:
importances_dt = dt_model.feature_importances_
n_components   = len(importances_dt)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Decision Tree importances
top_dt = np.argsort(importances_dt)[::-1][:15]
axes[0].bar(range(15), importances_dt[top_dt],
            color='indianred', edgecolor='white')
axes[0].set_xticks(range(15))
axes[0].set_xticklabels([f'PC{i+1}' for i in top_dt], rotation=45, ha='right')
axes[0].set_title('Decision Tree — Top 15 PC Importances')
axes[0].set_ylabel('Gini Importance')

# Random Forest importances
top_rf = np.argsort(importances_rf)[::-1][:15]
axes[1].bar(range(15), importances_rf[top_rf],
            color='mediumseagreen', edgecolor='white')
axes[1].set_xticks(range(15))
axes[1].set_xticklabels([f'PC{i+1}' for i in top_rf], rotation=45, ha='right')
axes[1].set_title('Random Forest — Top 15 PC Importances')
axes[1].set_ylabel('Gini Importance')

plt.tight_layout()
plt.show()

print("A single tree concentrates importance on the few PCs it happens to split")
print("on, while the forest spreads importance over many PCs (averaged across 100")
print("trees), which is why its importance ranking is more stable.")
print()
top5_dt = set([f'PC{i+1}' for i in top_dt[:5]])
top5_rf = set([f'PC{i+1}' for i in top_rf[:5]])
print(f"  Decision Tree top 5 : {sorted(top5_dt)}")
print(f"  Random Forest top 5 : {sorted(top5_rf)}")
print(f"  Common PCs          : {sorted(top5_dt & top5_rf)}")

---
## Part 4 — Advantages and Limitations: Single Tree vs Ensemble

### 4.1 Why the ensemble outperforms the single tree

The comparison above is the textbook illustration of the **bias–variance
trade-off**:

- A single decision tree has **low bias** but **high variance** — it fits the
  training set almost perfectly (often 100% training accuracy) yet generalises
  poorly, because the exact tree depends heavily on which samples it saw.
- Random Forest keeps the low bias of trees but **reduces variance** by averaging
  100 trees, each trained on a different bootstrap sample with a random feature
  subset at every split. The errors of individual trees partly cancel out.
- The diversity between trees is what makes the forest stronger than any one of
  its trees.

### 4.2 Advantages

| Advantage | Decision Tree | Random Forest |
|---|---|---|
| Interpretability (readable rules) | ✅ High | ❌ Black box (100 trees) |
| Training / prediction speed | ✅ Fast | ⚠️ Slower (100 trees) |
| Resistance to overfitting | ❌ Low | ✅ Higher (averaging) |
| Stable feature importance | ❌ From one tree only | ✅ Averaged over 100 trees |
| Handles p >> n (54K features, 56 samples) | ⚠️ Overfits easily | ✅ Better |
| Handles class imbalance (`class_weight`) | ✅ Yes | ✅ Yes |
| No distributional assumptions | ✅ Yes | ✅ Yes |

### 4.3 Limitations

| Limitation | Decision Tree | Random Forest |
|---|---|---|
| Variance / instability | ❌ High (small data change → different tree) | ✅ Reduced |
| Memory usage | ✅ Low (one tree) | ❌ Higher (stores 100 trees) |
| Interpretability | ✅ Full | ❌ Only via importances / surrogates |
| Gene-level interpretation | ⚠️ Only via PCA back-projection | ⚠️ Only via PCA back-projection |
| Sensitivity to small n | ❌ Severe overfitting | ⚠️ Bootstrap loses some samples |

### 4.4 Which is better for this dataset?

In [ ]:
print("=== Final Comparison Summary ===")
print()
print(f"{'Metric':<30} {'Decision Tree':>15} {'Random Forest':>15}")
print("-" * 62)
print(f"{'Training Accuracy':<30} {train_acc_dt:>15.4f} {train_acc_rf:>15.4f}")
print(f"{'CV Accuracy (mean)':<30} {dt_cv_acc.mean():>15.4f} {rf_cv_acc.mean():>15.4f}")
print(f"{'CV Accuracy (std)':<30} {dt_cv_acc.std():>15.4f} {rf_cv_acc.std():>15.4f}")
print(f"{'CV F1 weighted (mean)':<30} {dt_cv_f1.mean():>15.4f} {rf_cv_f1.mean():>15.4f}")
print(f"{'Test Accuracy':<30} {test_acc_dt:>15.4f} {test_acc_rf:>15.4f}")
print(f"{'Test F1 weighted':<30} {test_f1_dt:>15.4f} {test_f1_rf:>15.4f}")
print()

if rf_cv_acc.mean() >= dt_cv_acc.mean():
    winner = "Random Forest"
    reason = ("Averaging 100 bootstrapped trees reduces the high variance of a "
              "single tree, which matters a lot with only 56 samples and 54K "
              "features, where one tree overfits badly.")
else:
    winner = "Decision Tree"
    reason = ("On this particular split the single tree matches or beats the "
              "forest in CV, but note its large train/test gap — its high "
              "variance makes that result fragile.")

print(f"Best CV performer: {winner}")
print(f"Reason: {reason}")
print()
print("Important caveat: with only 12 test samples, test-set results have very")
print("high variance. Cross-validation is the more reliable estimate here.")

### 4.5 Written Analysis

#### Performance comparison

Both models were trained under identical conditions (same PCA, same split, same
class weighting) so that the only difference is **one fully-grown tree vs an
ensemble of 100 trees**.

The clearest signal is the **train/test gap**. The single Decision Tree reaches
near-100% training accuracy but drops substantially on the test set and across
CV folds — the signature of **high variance / overfitting**. The Random Forest
shows a smaller gap and a lower CV standard deviation, i.e. more stable
predictions, because averaging many trees cancels out much of that variance.

With only 56 samples the absolute scores stay modest for both models, and the CV
confidence intervals overlap — but the forest is the safer choice because it is
far less sensitive to the particular samples it was trained on.

#### Advantages of the ensemble over a single tree

1. **Lower variance**: a single tree on 56 samples overfits completely; averaging
   100 trees smooths out individual-tree errors.
2. **More stable feature importance**: importances averaged over 100 trees are far
   steadier than those read off one tree, which fixates on a handful of PCs.
3. **Consistent imbalance handling**: `class_weight='balanced'` propagates through
   every tree, giving steadier compensation for the minority class.

#### What the single Decision Tree still offers

1. **Interpretability**: the tree is fully readable (Part 3.1) — you can trace the
   exact rules behind a prediction, which a 100-tree forest cannot offer.
2. **Speed and low memory**: training and prediction are cheap, and only one tree
   is stored.
3. **A useful baseline**: it shows how much the ensemble actually buys you — if the
   forest barely beat the tree, the extra complexity would not be justified.

#### Limitations specific to this dataset

1. **Sample-size bottleneck**: with n=56, even the forest cannot fully compensate
   for the lack of data; both models show high variance across CV folds.
2. **PCA information loss**: reducing 54,630 probes to ~41 PCA components (95%
   variance) discards 5% of the signal for both models.
3. **Biological interpretability**: importances are expressed over PCA components,
   not individual genes; back-projecting through `pca.components_` is needed to
   recover specific biomarkers.

#### Conclusion

The single Decision Tree is interpretable and fast but overfits this small,
high-dimensional dataset. Random Forest keeps the trees' flexibility while taming
their variance through bagging and feature randomness, making it the more reliable
choice here — though model selection should lean on cross-validation rather than
the 12-sample test set. For future work, enlarging the dataset (e.g. merging
GSE6575 with GSE18123) would give a firmer basis for comparison.

In [ ]:
print("=== Phase 4 Complete ===")
print()
print("Summary:")
print(f"  Baseline model : single Decision Tree + PCA(95%)")
print(f"    Train Accuracy: {train_acc_dt:.4f}   (CV {dt_cv_acc.mean():.4f} +/- {dt_cv_acc.std():.4f})")
print(f"    Test  Accuracy: {test_acc_dt:.4f}")
print()
print(f"  Ensemble model : Random Forest + PCA(95%)")
print(f"    Train Accuracy: {train_acc_rf:.4f}   (CV {rf_cv_acc.mean():.4f} +/- {rf_cv_acc.std():.4f})")
print(f"    Test  Accuracy: {test_acc_rf:.4f}")
print()
print("A Random Forest is an ensemble of decision trees like the one above.")
print("The single tree overfits (large train/test gap); the forest reduces")
print("variance by averaging 100 trees trained on bootstrap samples.")